# T08 — Đo thông lượng và quyết định bậc thang

T07 đã trả lời "một mẫu có vừa 16 GB không". Notebook này trả lời câu quyết định lịch chạy:
**với 30 giờ GPU mỗi tuần, trích đặc trưng cho cả bốn bộ có kịp không.**

Cách làm: chia mẫu theo bốn mức độ dài, đo 20 mẫu mỗi mức, rồi nhân trung vị từng mức với
phân bố độ dài thật của từng bộ. Kết quả ra số giờ GPU cho mỗi bộ, không phải một con số
ms/mẫu chung chung.

**Notebook settings trước khi chạy:**

- Accelerator: **GPU T4 x2**
- Internet: **On**
- Data: attach dataset `unicorn1209/vihallulens` — bắt buộc, cần cả bốn bộ để dựng phân bố
- Add-ons → Secrets: `HF_TOKEN` (không bắt buộc)

Ước tính khoảng **10–15 phút GPU**: nạp mô hình 2 phút, tokenize 71.520 mẫu 1–2 phút, đo 80
mẫu 5–8 phút.

Chạy hết từ trên xuống rồi copy output của **ô 3, ô 4 và ô 5** dán vào PR.

In [1]:
# Ô 1 — lấy code. Chạy lại được nhiều lần: nếu thư mục đã có thì kéo bản mới về,
# vì `git clone` vào thư mục đã tồn tại sẽ hỏng và ta lặng lẽ chạy tiếp bằng code cũ.
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/wsunicorn/vihallulens.git"
REPO_DIR = Path("/kaggle/working/vihallulens")


def run(*args, cwd=None):
    done = subprocess.run(args, cwd=cwd, capture_output=True, text=True)
    if done.returncode:
        raise RuntimeError(" ".join(args) + chr(10) + done.stdout + done.stderr)
    return done.stdout.strip()


if (REPO_DIR / ".git").is_dir():
    run("git", "fetch", "--quiet", "origin", cwd=REPO_DIR)
    run("git", "reset", "--quiet", "--hard", "origin/main", cwd=REPO_DIR)
    print("đã cập nhật repo có sẵn")
else:
    run("git", "clone", "--quiet", REPO_URL, str(REPO_DIR))
    print("đã clone mới")

%cd /kaggle/working/vihallulens
print("commit:", run("git", "log", "--oneline", "-1", cwd=REPO_DIR))

đã clone mới
/kaggle/working/vihallulens
commit: c85101a T08: công cụ đo thông lượng và dự báo giờ GPU (#17)


In [2]:
# Ô 2 — cài đặt. Không cài lại torch: image Kaggle đã có bản dựng theo đúng CUDA của máy.
!pip install -q --no-deps -e .
!pip install -q -U bitsandbytes accelerate transformers pytest

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
ERROR: Package 'vihallulens' requires a different Python: 3.12.13 not in '<3.12,>=3.11'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 46.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 79.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.5/386.5 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 28.7 MB/s eta 0:00:00


In [3]:
# Ô 3 — kiểm tra môi trường và toán dự báo, trên CPU. Ô này hỏng thì DỪNG, đừng đốt quota.
# Chạy bằng tiến trình riêng chứ không import trong kernel: `pip install -e .` ghi một file
# .pth mà Python chỉ đọc lúc khởi động, nên kernel đang chạy sẵn có thể không thấy gói.
import os

try:
    from kaggle_secrets import UserSecretsClient

    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN: đã nạp từ Kaggle Secrets")
except Exception:
    print("HF_TOKEN: không có, vẫn chạy được vì Qwen2.5 là mô hình mở")

get_ipython().system("python scripts/probe_env.py")
get_ipython().system("python -m pytest tests/test_throughput.py -q")

HF_TOKEN: không có, vẫn chạy được vì Qwen2.5 là mô hình mở

MÔI TRƯỜNG
  repo             : /kaggle/working/vihallulens
  commit           : c85101a T08: công cụ đo thông lượng và dự báo giờ GPU (#17)
  python           : 3.12.13
  torch            : 2.10.0+cu128
  transformers     : 5.15.1
  bitsandbytes     : 0.50.1
  accelerate       : 1.14.0
  vihallulens      : 0.1.0 tại /kaggle/working/vihallulens/src/vihallulens/__init__.py
  dữ liệu          : /kaggle/input/datasets/unicorn1209/vihallulens  (14 file)
      MANIFEST.md
      isedsc01_test_private.json
      isedsc01_test_public.json
      isedsc01_train.json
      vifactcheck_dataset_card.md
      vifactcheck_dev.parquet
      vifactcheck_gitattributes.txt
      vifactcheck_test.parquet
      vifactcheck_train.parquet
      vihallu_test_public.csv
      vihallu_train.csv
      viwikifc_dev.csv
      viwikifc_test.csv
      viwikifc_train.csv
..........................                                               [100%]


In [4]:
# Ô 4 — thử khô: đọc đủ bốn bộ, dựng phân bố độ dài, chọn mẫu để đo. Không nạp mô hình.
# Nếu ô này ra bảng đúng thì mọi thứ ngoài GPU đã chạy được. Khoảng 2 phút.
# Copy output dán vào PR.
!python scripts/measure_throughput.py --dry-run


T08 — ĐO THÔNG LƯỢNG VÀ QUYẾT ĐỊNH BẬC THANG
  mô hình               : Qwen/Qwen2.5-7B-Instruct
  ngân sách token       : 4096
  compute dtype         : float16
  lớp bỏ qua            : [27]
  dữ liệu               : /kaggle/input/datasets/unicorn1209/vihallulens
config.json: 100%|█████████████████████████████| 663/663 [00:00<00:00, 1.94MB/s]
tokenizer_config.json: 7.30kB [00:00, 13.6MB/s]
vocab.json: 2.78MB [00:00, 45.8MB/s]
merges.txt: 1.67MB [00:00, 91.3MB/s]
tokenizer.json: 7.03MB [00:00, 113MB/s]
  chế độ                : thử khô, chỉ nạp tokenizer (8 s)
  khung mẫu prompt      : 37 token, 41 token khi có câu hỏi

--------------------------------------------------------------------------------
PHÂN BỐ ĐỘ DÀI THEO MỨC — đếm trên toàn bộ mẫu có nhãn
--------------------------------------------------------------------------------
  Bộ                     0–512      513–1024     1025–2048     2049–4096      tổng
  ---------------------------------------------------------------------

In [5]:
# Ô 5 — T08. Đo thật trên GPU rồi ghi vào results/feasibility.jsonl.
# Copy TOÀN BỘ output dán vào PR — phần KẾT LUẬN là căn cứ quyết định lịch chạy.
!python scripts/measure_throughput.py --per-tier 20


T08 — ĐO THÔNG LƯỢNG VÀ QUYẾT ĐỊNH BẬC THANG
  mô hình               : Qwen/Qwen2.5-7B-Instruct
  ngân sách token       : 4096
  compute dtype         : float16
  lớp bỏ qua            : [27]
  dữ liệu               : /kaggle/input/datasets/unicorn1209/vihallulens
model.safetensors.index.json: 27.8kB [00:00, 58.8MB/s]
Fetching 4 files: 100%|███████████████████████████| 4/4 [00:58<00:00, 14.65s/it]
Download complete: 100%|████████████████████| 15.2G/15.2G [00:58<00:00, 278MB/s]
Loading weights:  27%|██████▊                  | 92/339 [00:09<00:11, 22.26it/s]
Download complete: 100%|████████████████████| 15.2G/15.2G [01:09<00:00, 278MB/s]
Loading weights: 100%|████████████████████████| 339/339 [00:45<00:00,  7.45it/s]

generation_config.json: 100%|███████████████████| 243/243 [00:00<00:00, 850kB/s]
  nạp mô hình           : 116 s, 27 lớp được hook
  khung mẫu prompt      : 37 token, 41 token khi có câu hỏi

--------------------------------------------------------------------------------


In [6]:
# Ô 6 — chỉ chạy nếu ô 5 kết luận CẦN QUYẾT ĐỊNH.
# Nấc 1 của bảng sáu nấc lùi ở mục 5 CLAUDE.md: hạ ngân sách token còn 2.048.
# Đo ở T05: chỉ cắt thêm 1,09 % mẫu ISE-DSC01. Đây là số thật, không phải ước lượng.
!python scripts/measure_throughput.py --per-tier 20 --max-context-tokens 2048


T08 — ĐO THÔNG LƯỢNG VÀ QUYẾT ĐỊNH BẬC THANG
  mô hình               : Qwen/Qwen2.5-7B-Instruct
  ngân sách token       : 2048
  compute dtype         : float16
  lớp bỏ qua            : [27]
  dữ liệu               : /kaggle/input/datasets/unicorn1209/vihallulens
Loading weights: 100%|████████████████████████| 339/339 [00:19<00:00, 17.70it/s]
  nạp mô hình           : 31 s, 27 lớp được hook
  khung mẫu prompt      : 37 token, 41 token khi có câu hỏi

--------------------------------------------------------------------------------
PHÂN BỐ ĐỘ DÀI THEO MỨC — đếm trên toàn bộ mẫu có nhãn
--------------------------------------------------------------------------------
  Bộ                     0–512      513–1024     1025–2048     2049–4096      tổng
  --------------------------------------------------------------------------------
  vihallu                6,461           530             6             3     7,000
  isedsc01               6,188        17,770        11,931           480    3

In [7]:
# Ô 7 — lấy file kết quả về máy. results/ không được ghi ngược lên GitHub từ notebook,
# nên tải file này xuống rồi commit từ máy cá nhân.
import shutil

shutil.copy("results/feasibility.jsonl", "/kaggle/working/feasibility.jsonl")
with open("results/feasibility.jsonl", encoding="utf-8") as handle:
    print(handle.read())

{"config": {"extractor": {"compute_dtype": "float16", "device": "cuda", "exclude_layers": [27], "max_context_tokens": 4096, "model_name": "Qwen/Qwen2.5-7B-Instruct", "quantization": "nf4"}, "measurement": {"per_tier": 20, "sample_pool": ["vihallu", "isedsc01"], "tier_bounds": [512, 1024, 2048, 4096]}}, "config_hash": "1e47e41b73ee", "extra": {"gpu": "Tesla T4", "ms_per_sample": 742.1291515192838, "n_layers_hooked": 27, "n_samples_timed": 80, "peak_vram_mb": 8428.44384765625, "weekly_gpu_hours": 30.0}, "git_commit": "c85101a", "metrics": {"fits_half_weekly_quota": true, "length_scaling_exponent": 1.0006791597489166, "projected_hours": {"isedsc01": 9.50611314783438, "vifactcheck": 2.0704515741062837, "vihallu": 0.8175337920717554, "viwikifc": 2.3495339628373504}, "projected_hours_all": 14.743632476849772, "projected_hours_core": 10.323646939906135, "tier_histogram": {"isedsc01": {"0–512": 6188, "1025–2048": 11931, "2049–4096": 480, "513–1024": 17770}, "vifactcheck": {"0–512": 705, "1025–

## Đọc kết quả thế nào

Ba con số quyết định, theo thứ tự quan trọng:

1. **Giờ GPU cho hai bộ bắt buộc** (ViHallu + ISE-DSC01). Ngưỡng đặt ở **15 giờ**, tức nửa
   quota tuần, vì nửa còn lại phải dành cho E09 tinh chỉnh bộ mã hóa và các ablation.
2. **Mũ k** trong `chi phí ≈ độ dài^k`. Gần 2 nghĩa là ma trận chú ý chi phối, hạ ngân sách
   token xuống một nửa sẽ rẻ đi gần bốn lần. Gần 1 nghĩa là phần còn lại của mạng chi phối,
   nấc lùi 1 gần như không giúp gì và phải tính hướng khác.
3. **VRAM đỉnh theo mức**. Phải còn dư so với 14 GB ở mức dài nhất, vì đây mới là lúc chạy
   liên tiếp nhiều mẫu chứ không phải hai mẫu như T07.
4. **Bảng nhiệt độ và xung SM**. Nếu in ra `CẢNH BÁO HẠ XUNG` thì các mức đo sau đã chịu
   thiệt so với mức đo trước, và số của phiên này không so thẳng được với phiên khác.

Nếu ô 5 kết luận **CẦN QUYẾT ĐỊNH**, đừng tự chọn hướng: ghi số vào **Nhật ký chặn** cuối
`TASKS.md` rồi hỏi, theo quy tắc 4 mục 6 của `CLAUDE.md`. Lùi mô hình đọc chính là đổi quyết
định đã chốt ở mục 3 `CLAUDE.md`.